# Từ API đến bảng kết quả Chương 4 — một luận văn chạy được trong 3 phút

**Đề tài minh hoạ:** *Tác động của nợ xấu đến hiệu quả hoạt động của các ngân hàng
thương mại niêm yết trên HOSE và HNX.*

Notebook này không dạy `finlens`. Nó làm **đúng một luận văn định lượng**, từ đầu đến
cuối, trên dữ liệu thật:

> lấy mẫu → dựng dữ liệu bảng (panel) → thống kê mô tả → ma trận tương quan → VIF →
> Pooled OLS / FEM / REM → kiểm định F, Hausman, Breusch–Pagan LM → kiểm định khuyết
> tật → chọn mô hình → **đọc kết quả bằng lời**.

Phần kinh tế lượng giống hệt Stata. Thứ khác biệt là **bước tốn nhiều thời gian nhất
của một luận văn đã biến mất**: không tải file, không copy Excel, không gõ tay 297 dòng
số liệu, không phải kiểm tra lại xem mình có chép nhầm cột nào không.

| Cách làm quen thuộc | Notebook này |
|---|---|
| Tải BCTC từng ngân hàng, từng năm | 1 lời gọi API |
| Tự tính NPL từ thuyết minh nhóm nợ | `npl_ratio` tính sẵn, kèm công thức |
| Gõ GDP, CPI từ Tổng cục Thống kê | 1 lời gọi `macro.series` |
| Số liệu sửa một chỗ → chạy lại bằng tay | `Run All` — 3 phút |
| Kết quả khó tái lập | Cùng script, cùng bảng số |

---

## 1. Mô hình nghiên cứu

$$
\begin{aligned}
ROA_{it} &= \beta_0 + \beta_1 NPL_{it} + \beta_2 LDR_{it} + \beta_3 CAP_{it}
          + \beta_4 SIZE_{it} + \beta_5 CIR_{it} + \beta_6 GDPGR_t + \beta_7 INF_t
          + u_i + \varepsilon_{it} \\[2pt]
ROE_{it} &= \ldots \text{(cùng vế phải)} \\[2pt]
NIM_{it} &= \ldots \text{(cùng vế phải)}
\end{aligned}
$$

với $i$ là ngân hàng, $t$ là năm, $u_i$ là tác động riêng không quan sát được của từng
ngân hàng. Chính $u_i$ là thứ FEM và REM xử lý khác nhau — và Hausman là kiểm định phân
xử giữa hai cách ấy.

## 2. Bảng biến và dấu kỳ vọng

| Biến | Vai trò | Đo lường | Mã FinLens | Dấu kỳ vọng | Cơ sở lý thuyết |
|---|---|---|---|:--:|---|
| **ROA** | phụ thuộc | LNST / tổng tài sản | `roa` | — | |
| **ROE** | phụ thuộc | LNST / vốn chủ sở hữu | `roe` | — | |
| **NIM** | phụ thuộc | Thu nhập lãi thuần / tài sản sinh lãi | `nim` | — | |
| **NPL** | **độc lập chính** | Nợ nhóm 3+4+5 / dư nợ | `npl_ratio` | **(−)** | Berger & DeYoung (1997) |
| **CAP** | kiểm soát | CAR, hoặc VCSH / tổng TS | `car` · `equity_to_assets` | (+/−) | Đánh đổi an toàn vốn ↔ đòn bẩy |
| **LDR** | kiểm soát | Dư nợ / tiền gửi | `ldr` | (+) | Mức độ chuyển hoá vốn |
| **SIZE** | kiểm soát | ln(tổng tài sản) | `total_assets` | (+/−) | Lợi thế quy mô ↔ phi hiệu quả quy mô |
| **CIR** | kiểm soát | Chi phí HĐ / tổng thu nhập HĐ | `cir` | **(−)** | Hiệu quả quản trị chi phí |
| **GDPGR** | vĩ mô | Tăng trưởng GDP | `macro` | (+) | Chu kỳ kinh tế |
| **INF** | vĩ mô | Lạm phát CPI bình quân | `macro` | (+/−) | Hiệu ứng Fisher ↔ chi phí thực |

## 3. Yêu cầu

```bash
pip install finlens pandas plotly linearmodels
```

`linearmodels` kéo theo `statsmodels` và `scipy`. Tương ứng với Stata:
`PanelOLS(..., entity_effects=True)` ≡ `xtreg, fe` · `RandomEffects` ≡ `xtreg, re`.

In [ ]:
import sys
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, PooledOLS, RandomEffects, compare
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor

import finlens
from finlens_examples import ap_dung_theme, duong, heatmap
from finlens_examples.charts import CHUOI, GIAM, TANG

ap_dung_theme()
client = finlens.client()

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)

NAM_DAU, NAM_CUOI = 2015, 2025           # khung thời gian nghiên cứu
MUC_Y_NGHIA = 0.05                       # mức ý nghĩa dùng cho Hausman và các kiểm định

print(f"finlens {finlens.__version__} · pandas {pd.__version__}")
print(f"Khung phân tích: {NAM_DAU}–{NAM_CUOI} ({NAM_CUOI - NAM_DAU + 1} năm)")

---
## Bước 1 — Xác định mẫu nghiên cứu

Một luận văn bắt đầu bằng câu *"mẫu gồm các NHTM cổ phần niêm yết trên HOSE và HNX"*.
Câu đó phải trở thành một **danh sách mã cụ thể**, và danh sách ấy phải kiểm chứng được.

`8300` là mã ngành ICB *Banks*. Đừng gõ tay danh sách ngân hàng: gõ tay thì một năm sau
đọc lại, không ai biết vì sao thiếu một mã.

In [ ]:
nganh_ngan_hang = client.meta.symbols(icb="8300")

SAN = ["HOSE", "HNX"]
mau = nganh_ngan_hang[nganh_ngan_hang["exchange"].isin(SAN)].copy()
MA_NH = sorted(mau["symbol"])

print(f"Toàn ngành ICB 8300: {len(nganh_ngan_hang)} mã")
print(nganh_ngan_hang.groupby("exchange", observed=True)["symbol"].count().to_string())
print(f"\n→ Mẫu nghiên cứu (HOSE + HNX): {len(MA_NH)} ngân hàng")
print("  " + ", ".join(MA_NH))
print(f"\nLoại hình doanh nghiệp trong mẫu: {mau['company_type'].unique().tolist()}  (NH = ngân hàng)")
print(f"Cỡ mẫu tối đa: {len(MA_NH)} NH × {NAM_CUOI - NAM_DAU + 1} năm = "
      f"{len(MA_NH) * (NAM_CUOI - NAM_DAU + 1)} quan sát")

> **Viết vào Chương 3 thế nào.** Các ngân hàng trên UPCoM bị loại *theo tiêu chí niêm
> yết*, không phải vì thiếu số liệu — đó là một lựa chọn có lý do và phải nêu ra. Bảng
> `exchange` ở trên chính là bằng chứng cho tiêu chí ấy.

---
## Bước 2 — Lấy dữ liệu ngân hàng

`financials.indicators()` trả về các chỉ tiêu **đã tính sẵn** — không phải khoản mục thô
để mỗi người tự dựng một kiểu công thức.

Hai câu hội đồng chắc chắn hỏi, và cả hai đều trả lời được từ danh mục chỉ tiêu:

1. **Chỉ tiêu này định nghĩa thế nào?** → cột `formula`.
2. **Đơn vị là gì?** → cột `unit`. `ratio` nghĩa là phân số: `0.0167` = 1,67%.

In [ ]:
cat_nh = client.financials.indicator_catalog(com_type="NH").set_index("code")

MA_CHI_TIEU = ["roa", "roe", "nim", "npl_ratio", "ldr", "car",
               "equity_to_assets", "cir", "total_assets"]

thieu = [m for m in MA_CHI_TIEU if m not in cat_nh.index]
assert not thieu, f"Danh mục không có: {thieu} — kiểm tra lại tên mã trước khi gọi API"

for m in MA_CHI_TIEU:
    r = cat_nh.loc[m]
    print(f"■ {r['label']}  ·  {m}  [{r['unit']}]")
    print(f"   {r['formula']}\n")

In [ ]:
tho = client.financials.indicators(
    MA_NH, codes=MA_CHI_TIEU, period="annual", start_year=NAM_DAU, end_year=NAM_CUOI
)

print(f"{len(tho):,} dòng dạng long · {tho['symbol'].nunique()} mã · {tho['code'].nunique()} chỉ tiêu")
print("\nBa dòng đầu — mỗi dòng là MỘT chỉ tiêu của MỘT kỳ:")
print(tho.head(3)[["symbol", "year", "code", "value", "unit"]].to_string(index=False))

rong = tho.pivot_table(index=["symbol", "year"], columns="code", values="value", observed=True)
print(f"\nSau khi pivot: {rong.shape[0]} quan sát (NH × năm) × {rong.shape[1]} biến")

### Độ phủ dữ liệu — đọc trước khi chạy bất kỳ mô hình nào

Bảng dưới quyết định **mô hình viết ở Chương 3 có chạy được như đã viết hay không**.
Một biến phủ 40% không phải là biến kiểm soát; nó là cái bẫy làm bốc hơi hơn nửa số
quan sát ngay khi hồi quy, và không ai báo lỗi.

In [ ]:
so_o_toi_da = len(MA_NH) * (NAM_CUOI - NAM_DAU + 1)
do_phu = (rong.notna().sum() / so_o_toi_da * 100).sort_values()

bang_phu = pd.DataFrame({
    "chỉ tiêu": [cat_nh.loc[m, "label"] for m in do_phu.index],
    "số quan sát có": rong.notna().sum()[do_phu.index].to_numpy(),
    "độ phủ %": do_phu.round(1).to_numpy(),
})
print(f"Trên tổng {so_o_toi_da} ô (NH × năm):\n")
print(bang_phu.to_string(index=False))

### Quyết định thứ nhất: CAR hay ETA?

CAR là biến kiểm soát chuẩn trong nghiên cứu ngân hàng. Nhưng CAR **do từng ngân hàng
tự công bố**, và không phải ngân hàng nào cũng công bố đều đặn — nhất là giai đoạn
trước khi Basel II có hiệu lực (2020).

Thay vì lờ đi, ta để **độ phủ** quyết định, và ghi lại quyết định ấy:

- phủ ≥ 80% → dùng `car` đúng như thiết kế ở Chương 3;
- phủ < 80% → dùng **ETA = VCSH / Tổng tài sản** làm biến đại diện cho năng lực vốn,
  và **nêu rõ điều chỉnh này trong bài viết**.

Đây là tình huống thật của nghiên cứu thực nghiệm, không phải lỗi dữ liệu. Một hội đồng
sẽ đánh giá cao việc nêu ra hơn là việc giấu đi.

In [ ]:
NGUONG_PHU = 80.0
phu_car = float(do_phu.get("car", 0.0))

if phu_car >= NGUONG_PHU:
    BIEN_VON, TEN_VON = "car", "CAR"
    ghi_chu_von = f"CAR phủ {phu_car:.1f}% — dùng đúng như thiết kế ở Chương 3."
else:
    BIEN_VON, TEN_VON = "equity_to_assets", "ETA"
    ghi_chu_von = (
        f"CAR chỉ phủ {phu_car:.1f}% (< {NGUONG_PHU:.0f}%), không đủ để đưa vào hồi quy. "
        f"Thay bằng ETA = VCSH / Tổng tài sản — proxy phổ biến cho năng lực vốn. "
        f"ĐIỀU CHỈNH NÀY PHẢI ĐƯỢC NÊU TRONG CHƯƠNG 4 hoặc phần Hạn chế của đề tài."
    )

print(f"Biến năng lực vốn được chọn: {TEN_VON}  ({BIEN_VON})")
print(f"→ {ghi_chu_von}")

---
## Bước 3 — Hai biến vĩ mô

GDPGR và INF là biến **theo năm, không theo ngân hàng**: mọi ngân hàng trong cùng một
năm nhận cùng một giá trị. Chúng không giải thích được khác biệt *giữa* các ngân hàng,
nhưng vẫn cần có để tách phần biến động do chu kỳ kinh tế ra khỏi phần do nợ xấu.

Không gõ tay mã chỉ tiêu — tra trong danh mục rồi truyền mã đi.

In [ ]:
danh_muc_vm = client.macro.indicators()


def tim_ma(mo_ta: str, *, freq: str | None = None) -> str:
    """Tìm đúng MỘT mã chỉ tiêu vĩ mô khớp mô tả; ném lỗi nếu không có hoặc mơ hồ."""
    d = danh_muc_vm
    if freq:
        d = d[d["frequency"] == freq]
    d = d[d["name"].str.contains(mo_ta, case=False, na=False, regex=False)]
    if len(d) == 0:
        raise LookupError(f"Không tìm thấy chỉ tiêu nào khớp {mo_ta!r}")
    if len(d) > 1:
        raise LookupError(f"{len(d)} chỉ tiêu khớp {mo_ta!r}: {d['code'].tolist()[:5]}")
    return str(d["code"].iloc[0])


MA_GDP = tim_ma("Tổng GDP tăng trưởng YoY")
MA_CPI = tim_ma("Chỉ số giá tiêu dùng (So với cùng kỳ năm trước)")

vi_mo = client.macro.series([MA_GDP, MA_CPI], start=f"{NAM_DAU}-01-01", end=f"{NAM_CUOI}-12-31")
print(vi_mo.groupby("code", observed=True).agg(
    ten=("name", "first"), tan_suat=("frequency", "first"),
    don_vi=("unit", "first"), so_diem=("value", "count"),
).to_string())

In [ ]:
def theo_nam(ma: str) -> pd.Series:
    """Quy chuỗi quý/tháng về BÌNH QUÂN NĂM — đúng cách Việt Nam công bố 'lạm phát
    bình quân năm', và là xấp xỉ tốt của tăng trưởng GDP cả năm."""
    d = vi_mo[vi_mo["code"] == ma]
    s = d.assign(nam=d["date"].dt.year).groupby("nam")["value"].mean()
    if s.mean() > 50:                      # nguồn phát dạng chỉ số gốc 100 (vd 103,5)
        s = s - 100
        print(f"  ⚠️ {ma}: nguồn phát dạng chỉ số gốc 100 → đã trừ 100 để về % thay đổi")
    return s.round(3)


gdpgr = theo_nam(MA_GDP).rename("gdpgr")
inf = theo_nam(MA_CPI).rename("inf")

bang_vm = pd.concat([gdpgr, inf], axis=1)
bang_vm.index.name = "year"
print("\nHai biến vĩ mô theo năm (%):")
print(bang_vm.to_string())

---
## Bước 4 — Dựng dữ liệu bảng

Bốn việc, theo đúng thứ tự:

1. **Đổi đơn vị.** Mọi chỉ tiêu `unit = "ratio"` đang là phân số (`0.0167`). Nhân 100 để
   hệ số hồi quy đọc được thành *"điểm phần trăm"*. Đây là phép đổi thang tuyến tính:
   **không** làm đổi dấu, p-value, $R^2$, VIF hay bất kỳ kiểm định nào.
2. **SIZE = ln(tổng tài sản).** Tổng tài sản ở đơn vị **VND thô** và lệch phải rất
   mạnh — lấy log là chuẩn mực trong mọi nghiên cứu ngân hàng.
3. **Winsorize 1% / 99%.** Cắt đuôi hai phía để một quan sát dị biệt (ngân hàng bị kiểm
   soát đặc biệt, một năm lỗ nặng) không một mình kéo cả hệ số hồi quy. Ghi rõ trong
   Chương 3.
4. **Ghép biến vĩ mô** theo năm.

In [ ]:
don_vi = cat_nh["unit"].to_dict()
panel = rong.reset_index()

# (1) ratio → phần trăm
cot_ratio = [c for c in panel.columns if don_vi.get(c) == "ratio"]
panel[cot_ratio] = panel[cot_ratio] * 100
print(f"Đổi sang %: {cot_ratio}")

# (2) SIZE = ln(tổng tài sản)
panel["size"] = np.log(panel["total_assets"])

# (3) đổi tên theo ký hiệu luận văn
panel = panel.rename(columns={"npl_ratio": "npl", BIEN_VON: "von"})

BIEN_PHU_THUOC = {"roa": "ROA", "roe": "ROE", "nim": "NIM"}
BIEN_DOC_LAP = ["npl", "ldr", "von", "size", "cir", "gdpgr", "inf"]
TEN_HIEN = {"roa": "ROA", "roe": "ROE", "nim": "NIM", "npl": "NPL", "ldr": "LDR",
            "von": TEN_VON, "size": "SIZE", "cir": "CIR", "gdpgr": "GDPGR",
            "inf": "INF", "const": "Hằng số"}

# (4) ghép biến vĩ mô và sàn niêm yết
panel = panel.merge(bang_vm.reset_index(), on="year", how="left")
panel = panel.merge(mau[["symbol", "exchange"]], on="symbol", how="left")

print(f"\nPanel thô: {len(panel)} quan sát × {panel.shape[1]} cột")

In [ ]:
BIEN_CAT_DUOI = ["roa", "roe", "nim", "npl", "ldr", "von", "size", "cir"]
MUC_CAT = 0.01


def cat_duoi(s: pd.Series, muc: float = MUC_CAT) -> pd.Series:
    return s.clip(s.quantile(muc), s.quantile(1 - muc))


truoc = panel[BIEN_CAT_DUOI].copy()
panel[BIEN_CAT_DUOI] = panel[BIEN_CAT_DUOI].apply(cat_duoi)
so_bi_cat = int((truoc != panel[BIEN_CAT_DUOI]).sum().sum())

print(f"Winsorize {MUC_CAT:.0%}/{1 - MUC_CAT:.0%}: {so_bi_cat} giá trị bị kéo về ngưỡng "
      f"({so_bi_cat / int(truoc.notna().sum().sum()) * 100:.1f}% số ô có dữ liệu)")

In [ ]:
du_bien_x = panel.dropna(subset=BIEN_DOC_LAP)
print(f"Quan sát đủ toàn bộ biến độc lập: {len(du_bien_x)}/{len(panel)}")
print(f"Số ngân hàng còn lại: {du_bien_x['symbol'].nunique()}\n")

theo_nam_bang = du_bien_x.groupby("year", observed=True).agg(
    so_NH=("symbol", "nunique"),
    NPL_bq=("npl", "mean"),
    ROA_bq=("roa", "mean"),
    ROE_bq=("roe", "mean"),
    NIM_bq=("nim", "mean"),
).round(2)
print("Phân bố quan sát theo năm — panel có cân bằng không?")
print(theo_nam_bang.to_string())

can_bang = theo_nam_bang["so_NH"].nunique() == 1
print(f"\n→ Panel {'CÂN BẰNG' if can_bang else 'KHÔNG CÂN BẰNG'} "
      f"({theo_nam_bang['so_NH'].min()}–{theo_nam_bang['so_NH'].max()} NH mỗi năm)."
      f"{'' if can_bang else ' FEM/REM vẫn ước lượng được; nêu rõ trong Chương 3.'}")

### Xuất dữ liệu — để người dùng Stata / SPSS / EViews cũng chạy được

Đây là chỗ notebook này hữu ích cả với người **không** dùng Python: nó là một cỗ máy
dựng bộ dữ liệu luận văn. Chạy xong, mở file `.dta` bằng Stata rồi gõ
`xtset bank_id year` là đi tiếp được ngay.

In [ ]:
THU_MUC_RA = GOC / "output"
THU_MUC_RA.mkdir(exist_ok=True)

xuat = panel.copy()
xuat["bank_id"] = pd.factorize(xuat["symbol"])[0] + 1      # định danh số cho xtset
COT_XUAT = ["bank_id", "symbol", "exchange", "year", "roa", "roe", "nim",
            "npl", "ldr", "von", "size", "cir", "gdpgr", "inf"]
xuat = xuat[COT_XUAT].sort_values(["symbol", "year"])

duong_xlsx = THU_MUC_RA / "panel_ngan_hang_npl.xlsx"
duong_dta = THU_MUC_RA / "panel_ngan_hang_npl.dta"

with pd.ExcelWriter(duong_xlsx) as w:
    xuat.to_excel(w, sheet_name="panel", index=False)
    bang_phu.to_excel(w, sheet_name="do_phu_du_lieu", index=False)
    bang_vm.reset_index().to_excel(w, sheet_name="vi_mo", index=False)

xuat.to_stata(duong_dta, write_index=False, version=117)

print(f"✓ {duong_xlsx.name}  ({len(xuat)} dòng · 3 sheet)")
print(f'✓ {duong_dta.name}   → Stata:  use "{duong_dta.name}", clear   ·   xtset bank_id year')

---
## Bước 5 — Thống kê mô tả

Bảng này đi thẳng vào Chương 4. Nhưng nó không chỉ để trang trí: đọc **Std / Mean** và
khoảng **Min–Max** là biết ngay biến nào biến động đủ mạnh để có thể giải thích được
điều gì, và có quan sát nào bất thường còn sót lại sau winsorize hay không.

In [ ]:
CAC_BIEN = list(BIEN_PHU_THUOC) + BIEN_DOC_LAP


def bang_mo_ta(df: pd.DataFrame, cot: list[str]) -> pd.DataFrame:
    d = df[cot].agg(["count", "mean", "std", "min", "median", "max"]).T
    d["count"] = d["count"].astype(int)
    d.insert(0, "Biến", [TEN_HIEN[c] for c in d.index])
    return d.rename(columns={"count": "N", "mean": "Trung bình", "std": "Độ lệch chuẩn",
                             "min": "Nhỏ nhất", "median": "Trung vị", "max": "Lớn nhất"})


mo_ta = bang_mo_ta(du_bien_x, CAC_BIEN)
print("Đơn vị: % cho mọi biến, riêng SIZE là ln(VND)\n")
print(mo_ta.round(3).to_string(index=False))

In [ ]:
xu_huong = du_bien_x.groupby("year", observed=True)[["npl", "roa", "nim"]].mean().reset_index()
dai = xu_huong.melt(id_vars="year", var_name="bien", value_name="gia_tri")
dai["bien"] = dai["bien"].map({"npl": "NPL (%)", "roa": "ROA (%)", "nim": "NIM (%)"})

duong(
    dai,
    x="year",
    y="gia_tri",
    theo="bien",
    tieu_de=f"Nợ xấu và hiệu quả hoạt động toàn mẫu, {NAM_DAU}–{NAM_CUOI}",
    phu_de=f"Trung bình {du_bien_x['symbol'].nunique()} ngân hàng niêm yết HOSE + HNX · đơn vị %",
    nhan_y="%",
)

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=du_bien_x["npl"], y=du_bien_x["roa"], mode="markers",
        marker=dict(size=8, color=CHUOI[0], opacity=0.6, line=dict(width=1, color="#fcfcfb")),
        text=du_bien_x["symbol"] + " " + du_bien_x["year"].astype(str),
        hovertemplate="<b>%{text}</b><br>NPL %{x:.2f}%<br>ROA %{y:.2f}%<extra></extra>",
        name="quan sát",
    )
)
he_so = np.polyfit(du_bien_x["npl"], du_bien_x["roa"], 1)
truc_x = np.linspace(du_bien_x["npl"].min(), du_bien_x["npl"].max(), 50)
fig.add_trace(
    go.Scatter(x=truc_x, y=np.polyval(he_so, truc_x), mode="lines",
               line=dict(width=2, color=GIAM, dash="dash"),
               name=f"xu hướng: ROA = {he_so[1]:.3f} {he_so[0]:+.3f}·NPL")
)
fig.update_layout(
    title_text="Quan hệ NPL – ROA trên toàn mẫu<br>"
               "<sub style='color:#52514e'>Một điểm là một ngân hàng trong một năm · "
               "đường đứt nét chỉ là tương quan thô, chưa kiểm soát biến nào</sub>",
    xaxis_title="Tỷ lệ nợ xấu NPL (%)", yaxis_title="ROA (%)", height=520,
)
fig

---
## Bước 6 — Ma trận tương quan

Hai mục đích khác nhau, đừng trộn:

1. **Xem chiều quan hệ thô** giữa NPL và ba biến phụ thuộc — mới là *gợi ý*, chưa phải
   kết luận, vì chưa kiểm soát biến nào.
2. **Phát hiện đa cộng tuyến sơ bộ**: cặp biến độc lập nào có |r| > 0,8 là dấu hiệu
   cảnh báo. Nhưng tương quan cặp **không đủ** để kết luận — đó là việc của VIF ở bước
   sau, vì đa cộng tuyến có thể sinh ra từ tổ hợp nhiều biến chứ không chỉ từ một cặp.

In [ ]:
tuong_quan = du_bien_x[CAC_BIEN].corr()
n_qs = len(du_bien_x)

t_stat = tuong_quan * np.sqrt((n_qs - 2) / (1 - tuong_quan**2))
p_value = pd.DataFrame(2 * stats.t.sf(np.abs(t_stat), n_qs - 2),
                       index=tuong_quan.index, columns=tuong_quan.columns)


def sao(p: float) -> str:
    return "***" if p < 0.01 else ("**" if p < 0.05 else ("*" if p < 0.1 else ""))


hien = tuong_quan.copy().astype(object)
for i in tuong_quan.index:
    for j in tuong_quan.columns:
        hien.loc[i, j] = "1" if i == j else f"{tuong_quan.loc[i, j]:.3f}{sao(p_value.loc[i, j])}"
hien.index = [TEN_HIEN[c] for c in hien.index]
hien.columns = [TEN_HIEN[c] for c in hien.columns]

print(f"Ma trận tương quan Pearson · N = {n_qs} · *** p<0,01 · ** p<0,05 · * p<0,1\n")
print(hien.to_string())

cao = [(TEN_HIEN[i], TEN_HIEN[j], round(tuong_quan.loc[i, j], 3))
       for k, i in enumerate(BIEN_DOC_LAP) for j in BIEN_DOC_LAP[k + 1:]
       if abs(tuong_quan.loc[i, j]) > 0.8]
print(f"\nCặp biến độc lập có |r| > 0,8: {cao if cao else 'không có → chưa thấy dấu hiệu đa cộng tuyến nghiêm trọng'}")

In [ ]:
bang_nhiet = tuong_quan.round(2)
bang_nhiet.index = [TEN_HIEN[c] for c in bang_nhiet.index]
bang_nhiet.columns = [TEN_HIEN[c] for c in bang_nhiet.columns]

heatmap(
    bang_nhiet,
    tieu_de="Ma trận tương quan giữa các biến trong mô hình",
    phu_de="Đỏ là tương quan âm, xanh là dương · điểm giữa khoá ở 0 nên ô xám luôn nghĩa là 'gần như không liên hệ'",
    nhan_mau="hệ số r",
    dinh_dang_o="%{z:.2f}",
)

---
## Bước 7 — Kiểm định đa cộng tuyến (VIF)

Quy ước thường dùng trong luận văn:

| VIF | Kết luận |
|---|---|
| < 2 | không có dấu hiệu đa cộng tuyến |
| 2 – 5 | chấp nhận được |
| 5 – 10 | cần lưu ý |
| > 10 | đa cộng tuyến nghiêm trọng, phải xử lý |

VIF tính trên **các biến độc lập với nhau**, nên chỉ có **một** bảng VIF cho cả ba mô
hình ROA / ROE / NIM — không phải ba bảng.

In [ ]:
X_vif = sm.add_constant(du_bien_x[BIEN_DOC_LAP])
bang_vif = pd.DataFrame({
    "Biến": [TEN_HIEN[c] for c in BIEN_DOC_LAP],
    "VIF": [variance_inflation_factor(X_vif.to_numpy(), i + 1) for i in range(len(BIEN_DOC_LAP))],
})
bang_vif["Kết luận"] = pd.cut(
    bang_vif["VIF"], [0, 2, 5, 10, np.inf],
    labels=["không có", "chấp nhận được", "cần lưu ý", "nghiêm trọng"],
)
print(bang_vif.round(3).to_string(index=False))

vif_max = bang_vif["VIF"].max()
print(f"\nVIF lớn nhất = {vif_max:.2f} → "
      + ("không có đa cộng tuyến nghiêm trọng, giữ nguyên mô hình."
         if vif_max < 10 else "PHẢI xử lý: bỏ bớt biến hoặc thay bằng biến đại diện khác."))

---
## Bước 8 — Hồi quy Pooled OLS, FEM, REM và các kiểm định lựa chọn

Ba mô hình, ba giả định khác nhau về $u_i$ — tác động riêng không quan sát được của
từng ngân hàng (văn hoá quản trị rủi ro, khẩu vị rủi ro, chất lượng ban điều hành…):

| Mô hình | Giả định về $u_i$ | Hệ quả |
|---|---|---|
| **Pooled OLS** | không tồn tại $u_i$ | mọi ngân hàng giống nhau — gần như luôn bị bác bỏ |
| **FEM** | $u_i$ tồn tại và **có tương quan** với biến độc lập | ước lượng vẫn vững, nhưng mất biến không đổi theo thời gian |
| **REM** | $u_i$ tồn tại nhưng **không tương quan** với biến độc lập | hiệu quả hơn *nếu* giả định đúng |

Ba kiểm định phân xử, chạy theo đúng thứ tự này:

1. **F-test** (FEM vs Pooled OLS) — $H_0$: mọi $u_i = 0$. Bác bỏ → có tác động riêng, bỏ Pooled.
2. **Breusch–Pagan LM** (REM vs Pooled OLS) — $H_0$: $\sigma^2_u = 0$. Bác bỏ → bỏ Pooled.
3. **Hausman** (FEM vs REM) — $H_0$: $u_i$ **không** tương quan với biến độc lập, tức REM
   là nhất quán và hiệu quả. **Bác bỏ ($p < 0{,}05$) → chọn FEM. Không bác bỏ → chọn REM.**

Cây quyết định mà notebook áp dụng, và bạn chép nguyên vào Chương 3:

- **F và LM đều không bác bỏ** → không có tác động riêng → **Pooled OLS**;
- ngược lại → Hausman phân xử: **p < 0,05 → FEM**, **p ≥ 0,05 → REM**.

> ⚠️ Hausman phải tính trên hai ước lượng **chưa hiệu chỉnh sai số chuẩn** (không robust,
> không clustered), vì công thức dựa trên tính hiệu quả của REM dưới $H_0$. Hiệu chỉnh
> robust chỉ áp dụng **sau khi** đã chọn xong mô hình.

In [ ]:
def chuan_bi(y: str) -> pd.DataFrame:
    """Loại bỏ theo dòng (listwise deletion) đúng như Stata, rồi đặt chỉ mục (mã, năm)."""
    d = panel.dropna(subset=[y] + BIEN_DOC_LAP).copy()
    return d.set_index(["symbol", "year"]).sort_index()


def uoc_luong(y: str) -> dict:
    """Ước lượng cả ba mô hình cho một biến phụ thuộc."""
    d = chuan_bi(y)
    X = sm.add_constant(d[BIEN_DOC_LAP])
    return {
        "du_lieu": d,
        "pooled": PooledOLS(d[y], X).fit(),
        "fem": PanelOLS(d[y], X, entity_effects=True).fit(),
        "rem": RandomEffects(d[y], X).fit(),
    }


def hausman(fem, rem) -> tuple[float, int, float]:
    """Kiểm định Hausman: H0 = REM nhất quán (u_i không tương quan với biến độc lập)."""
    chung = [c for c in fem.params.index if c in rem.params.index and c != "const"]
    b = (fem.params[chung] - rem.params[chung]).to_numpy()
    V = (fem.cov.loc[chung, chung] - rem.cov.loc[chung, chung]).to_numpy()
    chi2 = float(b @ np.linalg.pinv(V) @ b)          # pinv: V có thể suy biến trong mẫu nhỏ
    bac_tu_do = int(np.linalg.matrix_rank(V))
    return chi2, bac_tu_do, float(stats.chi2.sf(chi2, bac_tu_do))


def breusch_pagan_lm(pooled) -> tuple[float, float]:
    """LM của Breusch–Pagan (dạng Baltagi–Li cho panel không cân bằng): H0 = var(u_i) = 0."""
    e = pooled.resids.rename("e").reset_index()
    e.columns = ["dv", "tg", "e"]
    nhom = e.groupby("dv")["e"]
    tong_binh_phuong_nhom = float((nhom.sum() ** 2).sum())
    tong_binh_phuong = float((e["e"] ** 2).sum())
    T_i = nhom.size()
    sigma2 = tong_binh_phuong / len(e)
    lm = (tong_binh_phuong_nhom - tong_binh_phuong) ** 2 / (2 * sigma2**2 * float((T_i * (T_i - 1)).sum()))
    return float(lm), float(stats.chi2.sf(lm, 1))

In [ ]:
ket_qua = {}
tom_tat = []

for y, nhan in BIEN_PHU_THUOC.items():
    kq = uoc_luong(y)
    chi2_h, df_h, p_h = hausman(kq["fem"], kq["rem"])
    lm, p_lm = breusch_pagan_lm(kq["pooled"])
    f_pooled = kq["fem"].f_pooled

    # Cây quyết định đầy đủ: F và LM đều không bác bỏ → Pooled OLS;
    # ngược lại Hausman phân xử giữa FEM và REM.
    if float(f_pooled.pval) >= MUC_Y_NGHIA and p_lm >= MUC_Y_NGHIA:
        chon = "Pooled OLS"
    else:
        chon = "FEM" if p_h < MUC_Y_NGHIA else "REM"

    kq.update(hausman=(chi2_h, df_h, p_h), lm=(lm, p_lm), chon=chon)
    ket_qua[y] = kq

    d = kq["du_lieu"]
    tom_tat.append({
        "Biến phụ thuộc": nhan,
        "N": int(kq["fem"].nobs),
        "Số NH": d.index.get_level_values(0).nunique(),
        "F (FEM vs Pooled)": round(float(f_pooled.stat), 3),
        "p (F)": f"{float(f_pooled.pval):.4f}",
        "LM (REM vs Pooled)": round(lm, 3),
        "p (LM)": f"{p_lm:.4f}",
        "Hausman χ²": round(chi2_h, 3),
        "df": df_h,
        "p (Hausman)": f"{p_h:.4f}",
        "Mô hình chọn": kq["chon"],
    })

bang_chon = pd.DataFrame(tom_tat)
print("KIỂM ĐỊNH LỰA CHỌN MÔ HÌNH\n")
print(bang_chon.to_string(index=False))
print(f"\nQuy tắc: p(F) ≥ {MUC_Y_NGHIA} VÀ p(LM) ≥ {MUC_Y_NGHIA} → Pooled OLS;"
      f" ngược lại p(Hausman) < {MUC_Y_NGHIA} → FEM, còn lại → REM")

### So sánh ba mô hình cho biến phụ thuộc chính (ROA)

`compare()` in ra đúng dạng bảng ba cột quen thuộc trong các luận văn — giá trị trong
ngoặc là **thống kê t**.

In [ ]:
print(compare({
    "Pooled OLS": ket_qua["roa"]["pooled"],
    "FEM": ket_qua["roa"]["fem"],
    "REM": ket_qua["roa"]["rem"],
}))

---
## Bước 9 — Kiểm định khuyết tật của mô hình

Chọn xong FEM hay REM **chưa phải là xong**. Hai khuyết tật phổ biến trong dữ liệu bảng
tài chính, và cả hai đều **không** làm hệ số bị chệch — chúng làm **sai số chuẩn sai**,
nghĩa là p-value sai, nghĩa là kết luận "có ý nghĩa thống kê" có thể sai:

- **Phương sai sai số thay đổi theo đơn vị** (kiểm định Wald hiệu chỉnh — tương đương
  `xttest3` trong Stata). Rất hay gặp: một ngân hàng có tổng tài sản 2 triệu tỷ và một
  ngân hàng 50 nghìn tỷ không thể có cùng phương sai phần dư.
- **Tự tương quan bậc 1** (kiểm định Wooldridge — tương đương `xtserial`). Cũng rất hay
  gặp: ROA năm nay dính chặt ROA năm ngoái.

Cách xử lý chuẩn mực: giữ nguyên mô hình đã chọn, nhưng dùng **sai số chuẩn vững nhóm
theo ngân hàng** (cluster-robust) — tương đương `xtreg …, fe vce(cluster bank_id)`.

In [ ]:
def wald_phuong_sai(fem) -> tuple[float, int, float]:
    """Wald hiệu chỉnh (Greene): H0 = phương sai sai số đồng nhất giữa các ngân hàng."""
    e = fem.resids.rename("e").reset_index()
    e.columns = ["dv", "tg", "e"]
    nhom = e.groupby("dv")["e"]
    sigma2_i = nhom.apply(lambda s: float((s**2).mean()))
    sigma2 = float((e["e"] ** 2).mean())
    V_i = nhom.apply(lambda s: float(((s**2 - (s**2).mean()) ** 2).sum() / (len(s) * (len(s) - 1)))
                     if len(s) > 1 else np.nan)
    dung = V_i.notna() & (V_i > 0)
    W = float((((sigma2_i - sigma2) ** 2) / V_i)[dung].sum())
    g = int(dung.sum())
    return W, g, float(stats.chi2.sf(W, g))


def wooldridge_ar1(y: str) -> tuple[float, float, float]:
    """Wooldridge (2002): H0 = không có tự tương quan bậc 1 trong phần dư."""
    d = panel.dropna(subset=[y] + BIEN_DOC_LAP).sort_values(["symbol", "year"]).copy()
    for c in [y] + BIEN_DOC_LAP:
        d["d_" + c] = d.groupby("symbol")[c].diff()
    d = d.dropna(subset=["d_" + c for c in [y] + BIEN_DOC_LAP])
    mo_hinh = sm.OLS(d["d_" + y], d[["d_" + c for c in BIEN_DOC_LAP]]).fit()
    d = d.assign(e=mo_hinh.resid)
    d["e_tre"] = d.groupby("symbol")["e"].shift(1)
    d = d.dropna(subset=["e_tre"])
    phu = sm.OLS(d["e"], d[["e_tre"]]).fit(cov_type="cluster", cov_kwds={"groups": d["symbol"]})
    b, se = float(phu.params.iloc[0]), float(phu.bse.iloc[0])
    F = ((b + 0.5) / se) ** 2                        # H0: hệ số = -0,5
    g = d["symbol"].nunique()
    return b, F, float(stats.f.sf(F, 1, g - 1))


hang = []
for y, nhan in BIEN_PHU_THUOC.items():
    W, g, p_W = wald_phuong_sai(ket_qua[y]["fem"])
    rho, F_w, p_w = wooldridge_ar1(y)
    hang.append({
        "Biến phụ thuộc": nhan,
        "Wald χ²": round(W, 1),
        "p (Wald)": f"{p_W:.4f}",
        "Phương sai thay đổi": "CÓ" if p_W < MUC_Y_NGHIA else "không",
        "Wooldridge F": round(F_w, 3),
        "p (Wooldridge)": f"{p_w:.4f}",
        "Tự tương quan": "CÓ" if p_w < MUC_Y_NGHIA else "không",
    })

bang_khuyet_tat = pd.DataFrame(hang)
print("KIỂM ĐỊNH KHUYẾT TẬT\n")
print(bang_khuyet_tat.to_string(index=False))
print("\n→ Có bất kỳ ô 'CÓ' nào thì sai số chuẩn thông thường không dùng được nữa:")
print("  ước lượng lại mô hình đã chọn với sai số chuẩn vững nhóm theo ngân hàng.")

---
## Bước 10 — Mô hình phù hợp nhất và bảng kết quả cuối cùng

Đây là **Bảng 4.x** của luận văn: ba cột, mỗi cột một biến phụ thuộc, mỗi ô là hệ số kèm
mức ý nghĩa và sai số chuẩn trong ngoặc.

In [ ]:
def uoc_luong_vung(y: str):
    """Ước lượng lại mô hình ĐÃ CHỌN với sai số chuẩn vững nhóm theo ngân hàng."""
    d = ket_qua[y]["du_lieu"]
    X = sm.add_constant(d[BIEN_DOC_LAP])
    tuy_chon = dict(cov_type="clustered", cluster_entity=True)
    chon = ket_qua[y]["chon"]
    if chon == "FEM":
        return PanelOLS(d[y], X, entity_effects=True).fit(**tuy_chon)
    if chon == "REM":
        return RandomEffects(d[y], X).fit(**tuy_chon)
    return PooledOLS(d[y], X).fit(**tuy_chon)


cot_bang, dong_thong_tin = {}, {}
for y, nhan in BIEN_PHU_THUOC.items():
    res = uoc_luong_vung(y)
    ket_qua[y]["cuoi"] = res
    cot_bang[nhan] = pd.Series(
        {TEN_HIEN[b]: f"{res.params[b]:>8.4f}{sao(res.pvalues[b]):<3}  ({res.std_errors[b]:.4f})"
         for b in res.params.index}
    )
    r2 = res.rsquared_within if ket_qua[y]["chon"] == "FEM" else res.rsquared
    dong_thong_tin[nhan] = pd.Series({
        "Mô hình": ket_qua[y]["chon"] + " (SE vững nhóm)",
        "Số quan sát": f"{int(res.nobs)}",
        "Số ngân hàng": f"{ket_qua[y]['du_lieu'].index.get_level_values(0).nunique()}",
        "R²": f"{float(r2):.4f}",
        "p (Hausman)": f"{ket_qua[y]['hausman'][2]:.4f}",
    })

bang_cuoi = pd.concat([pd.DataFrame(cot_bang), pd.DataFrame(dong_thong_tin).fillna("")])
print("BẢNG 4.x — KẾT QUẢ HỒI QUY MÔ HÌNH PHÙ HỢP NHẤT")
print("Hệ số · *** p<0,01 ** p<0,05 * p<0,1 · sai số chuẩn vững nhóm theo ngân hàng trong ngoặc\n")
print(bang_cuoi.to_string())

### Đọc kết quả — biến chính NPL

Đoạn dưới **sinh ra từ chính con số vừa ước lượng**, không phải viết tay. Đây là mẫu
diễn giải để chép vào Chương 4 rồi bổ sung phần đối chiếu với các nghiên cứu trước.

In [ ]:
def muc_y_nghia(p: float) -> str:
    if p < 0.01:
        return "có ý nghĩa thống kê ở mức 1%"
    if p < 0.05:
        return "có ý nghĩa thống kê ở mức 5%"
    if p < 0.1:
        return "có ý nghĩa thống kê ở mức 10%"
    return "KHÔNG có ý nghĩa thống kê"


print("=" * 78)
for y, nhan in BIEN_PHU_THUOC.items():
    res, d = ket_qua[y]["cuoi"], ket_qua[y]["du_lieu"]
    b, p = float(res.params["npl"]), float(res.pvalues["npl"])
    tb = float(d[y].mean())
    chieu = "giảm" if b < 0 else "tăng"

    print(f"\n■ {nhan} — mô hình {ket_qua[y]['chon']}, N = {int(res.nobs)}")
    print(f"  Hệ số NPL = {b:+.4f} · p = {p:.4f} → {muc_y_nghia(p)}.")
    if p < 0.1:
        print(f"  Diễn giải: tỷ lệ nợ xấu tăng thêm 1 điểm phần trăm thì {nhan} {chieu} trung bình")
        print(f"  {abs(b):.4f} điểm phần trăm, khi các yếu tố khác không đổi — tương đương "
              f"{abs(b) / tb * 100:.1f}% mức {nhan} bình quân của mẫu ({tb:.2f}%).")
        print(f"  Dấu {'ÂM — ĐÚNG' if b < 0 else 'DƯƠNG — NGƯỢC'} với kỳ vọng lý thuyết.")
    else:
        print(f"  Diễn giải: chưa đủ bằng chứng thống kê cho thấy nợ xấu tác động tới {nhan}")
        print(f"  trong mẫu này. Lưu ý: 'không có ý nghĩa' KHÁC 'không có tác động'.")

    y_nghia = [c for c in BIEN_DOC_LAP if c != "npl" and float(res.pvalues[c]) < 0.05]
    if y_nghia:
        danh_sach = ", ".join(
            f"{TEN_HIEN[c]} ({float(res.params[c]):+.4f}{sao(float(res.pvalues[c]))})" for c in y_nghia
        )
        print(f"  Biến kiểm soát có ý nghĩa ở mức 5%: {danh_sach}")
    else:
        print("  Không biến kiểm soát nào có ý nghĩa ở mức 5%.")
print("\n" + "=" * 78)

In [ ]:
he_so_npl = pd.DataFrame({
    "nhan": [f"{nhan} ({ket_qua[y]['chon']})" for y, nhan in BIEN_PHU_THUOC.items()],
    "he_so": [float(ket_qua[y]["cuoi"].params["npl"]) for y in BIEN_PHU_THUOC],
    "p": [float(ket_qua[y]["cuoi"].pvalues["npl"]) for y in BIEN_PHU_THUOC],
})

fig = go.Figure()
for _, r in he_so_npl.iterrows():
    res = ket_qua[[k for k, v in BIEN_PHU_THUOC.items() if r["nhan"].startswith(v)][0]]["cuoi"]
    lo, hi = res.conf_int().loc["npl"]
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[r["nhan"], r["nhan"]], mode="lines",
        line=dict(width=3, color="#898781"), showlegend=False,
        hovertemplate="khoảng tin cậy 95%: [%{x:.4f}]<extra></extra>",
    ))
fig.add_trace(go.Scatter(
    x=he_so_npl["he_so"], y=he_so_npl["nhan"], mode="markers+text",
    marker=dict(size=13, color=[GIAM if v < 0 else TANG for v in he_so_npl["he_so"]],
                line=dict(width=2, color="#fcfcfb")),
    text=[f"{v:+.4f}{sao(p)}" for v, p in zip(he_so_npl["he_so"], he_so_npl["p"])],
    textposition="top center", textfont=dict(size=11, color="#52514e"),
    showlegend=False, hovertemplate="hệ số %{x:.4f}<extra></extra>",
))
fig.add_vline(x=0, line_width=1, line_color="#898781")
fig.update_layout(
    title_text="Tác động của nợ xấu lên ba thước đo hiệu quả<br>"
               "<sub style='color:#52514e'>Chấm là hệ số, thanh ngang là khoảng tin cậy 95% · "
               "thanh cắt qua đường 0 nghĩa là chưa có ý nghĩa thống kê</sub>",
    xaxis_title="Thay đổi của biến phụ thuộc khi NPL tăng 1 điểm phần trăm (đvt: điểm %)",
    height=420,
)
fig

---
## Xuất kết quả ra file để dán vào luận văn

In [ ]:
duong_kq = THU_MUC_RA / "ket_qua_hoi_quy_npl.xlsx"

with pd.ExcelWriter(duong_kq) as w:
    mo_ta.round(4).to_excel(w, sheet_name="4.1_thong_ke_mo_ta", index=False)
    hien.to_excel(w, sheet_name="4.2_tuong_quan")
    bang_vif.round(4).to_excel(w, sheet_name="4.3_VIF", index=False)
    bang_chon.to_excel(w, sheet_name="4.4_lua_chon_mo_hinh", index=False)
    bang_khuyet_tat.to_excel(w, sheet_name="4.5_khuyet_tat", index=False)
    bang_cuoi.to_excel(w, sheet_name="4.6_ket_qua_hoi_quy")
    for y, nhan in BIEN_PHU_THUOC.items():
        res = ket_qua[y]["cuoi"]
        pd.DataFrame({
            "Biến": [TEN_HIEN[b] for b in res.params.index],
            "Hệ số": res.params.to_numpy(),
            "Sai số chuẩn": res.std_errors.to_numpy(),
            "t": res.tstats.to_numpy(),
            "p-value": res.pvalues.to_numpy(),
            "KTC 95% dưới": res.conf_int()["lower"].to_numpy(),
            "KTC 95% trên": res.conf_int()["upper"].to_numpy(),
        }).round(6).to_excel(w, sheet_name=f"chi_tiet_{nhan}", index=False)

print(f"✓ {duong_kq.name} — 6 bảng chuẩn Chương 4 + 3 sheet chi tiết hệ số")
print(f"  Thư mục: {THU_MUC_RA}")

---
## Kết luận và những gì hội đồng sẽ hỏi

**Về kết quả.** Đọc lại ba dòng hệ số NPL ở trên. Nếu dấu âm và có ý nghĩa với ROA và
ROE nhưng không có ý nghĩa với NIM, đó là một kết quả **có nội dung kinh tế**, không
phải mô hình hỏng: nợ xấu đánh vào lợi nhuận ròng qua chi phí trích lập dự phòng, trong
khi NIM được tính trên thu nhập lãi thuần — tức là *trước* dự phòng. Đúng một câu đó
thôi cũng đủ cho một đoạn thảo luận.

**Ba hạn chế phải tự nêu trước khi bị hỏi.**

1. **Nội sinh hai chiều.** Nợ xấu làm giảm lợi nhuận, nhưng ngân hàng lợi nhuận thấp
   cũng có động cơ che giấu nợ xấu và cho vay dưới chuẩn. FEM không xử lý được điều này.
   Hướng khắc phục: ước lượng **GMM hệ thống (Arellano–Bond / Blundell–Bond)** với
   $NPL_{t-1}$ làm biến công cụ.
2. **Nợ xấu có độ trễ.** Khoản vay xấu đi hôm nay thường chỉ hiện trên báo cáo sau vài
   quý. Thử thêm $NPL_{t-1}$ vào mô hình.
3. **Cơ cấu lại nợ và chuẩn phân loại thay đổi.** Thông tư 01/2020 và các văn bản gia
   hạn cho phép giữ nguyên nhóm nợ, nên NPL công bố giai đoạn 2020–2023 thấp hơn bản
   chất. Hướng xử lý: thêm biến giả giai đoạn, hoặc kiểm định vững với **nợ nhóm 2**
   (`group2_ratio`) làm thước đo thay thế.

**Bốn hướng mở rộng — đủ để thành bốn đề tài khác nhau.**

| Hướng | Thay đổi trong notebook |
|---|---|
| Thêm biến kiểm soát | Thêm mã vào `MA_CHI_TIEU` — danh mục có 31 chỉ tiêu cho ngân hàng |
| Dữ liệu quý thay vì năm | `period="quarterly"` — cỡ mẫu gấp 4, bắt được chu kỳ tốt hơn |
| Kiểm định vững | Đổi `npl_ratio` → `group2_ratio` hoặc `npl_coverage`, chạy lại toàn bộ |
| Mô hình động (GMM) | Thêm `roa_{t-1}` vào vế phải, dùng `linearmodels` hoặc `pydynpd` |

**Điều cuối cùng, và là điều đáng nói nhất trong một buổi workshop:** toàn bộ notebook
này chạy lại được. Sang năm, đổi `NAM_CUOI = 2026` rồi `Run All` — có bộ dữ liệu mới,
bảng mới, kết quả mới. Mỗi khoá học viên một bộ số liệu cập nhật, cùng một khung phân
tích. Đó là thứ một file Excel chép tay không làm được.